# 05 · QLoRA fine-tune

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WKS-DS/SlangSense/blob/main/notebooks/05_lora_finetune.ipynb)

Fine-tunes Qwen2.5-1.5B-Instruct, the model from notebook 04, on the 30k training posts. It learns to answer the notebook 04 zero-shot prompt with the post's label. Two short convergence tests run first, to check that training works before the full run. Then the fine-tuned model labels the 300 test posts, and `src/evaluate.py` scores them next to every other model.

- **Runs on:** Google Colab with a T4 GPU (*Runtime → Change runtime type → T4 GPU*). It can't run on the Mac, because 4-bit loading (`bitsandbytes`) needs an NVIDIA GPU.
- **Inputs:** `data/processed/train.csv`, `val.csv` and the test set, plus the code in `src/`, all cloned from GitHub
- **Outputs**, saved to Google Drive in `MyDrive/SlangSense/`:
  - `lora_adapter/`: the trained adapter. Copy it to `models.nosync/lora_adapter/` (it is not committed).
  - `preds_llm_lora.csv`, `replies_llm_lora.csv` and `lora_log.csv` (training and validation loss). Copy them to `results/`.
- **Time:** about 1 to 1.5 hours, mostly the full training run. The short test run prints a closer estimate. Checkpoints are saved to Drive, so if Colab disconnects, run the notebook again and training resumes where it stopped.

## Setup

Check that Colab gave us a GPU:

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the project code and data from GitHub (skipped if this session already has them)
![ -d /content/SlangSense ] || git clone --depth 1 https://github.com/WKS-DS/SlangSense.git /content/SlangSense
%cd /content/SlangSense

# Training packages, pinned so the run can be repeated. transformers is the same version as on the Mac.
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 bitsandbytes==0.50.2

In [ ]:
import gc
import math
import shutil
import sys
import time
import warnings
from pathlib import Path

import datasets
import huggingface_hub
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils import logging as transformers_logging
from trl import SFTConfig, SFTTrainer

# Let the notebook import our shared code from src/
ROOT = Path("/content/SlangSense")
sys.path.append(str(ROOT))
from src.evaluate import RESULTS_DIR, evaluate, load_test, save_replies, score_table
from src.labels import LABELS, parse_reply
from src.prompts import ask_llm, build_messages, use_greedy_decoding

# Keep the output short and readable on GitHub: no login notices, and no download or
# data-processing progress bars (Colab draws those as widgets, which GitHub can't display)
warnings.filterwarnings("ignore", message=".*HF_TOKEN")
huggingface_hub.utils.logging.set_verbosity_error()
huggingface_hub.utils.disable_progress_bars()
datasets.disable_progress_bars()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # the same model as notebook 04
SEED = 42

Checkpoints and outputs go to Google Drive, because Colab deletes its own files when the session ends. Colab asks for permission the first time.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/SlangSense")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## 1. How the fine-tune works

**Fine-tuning** means training a model further on our own examples. Each example here is the notebook 04 zero-shot prompt with one training post, and the answer is that post's label. Training nudges the model's weights so the right answer becomes more likely.

**LoRA (Low-Rank Adaptation).** Updating all 1.5 billion weights would need far more GPU memory than a free T4 has. LoRA freezes the original weights and adds pairs of small new matrices (the **adapter**) next to the model's layers. Only the adapter trains, about 1% of the size of the model. The trained adapter is a small file that is loaded on top of the unchanged base model. The **rank** (`r`) sets the size of the small matrices: a higher rank can learn more, but trains more weights.

**4-bit quantization.** Normally each weight is a 16-bit number. Storing the frozen weights in 4 bits cuts the model from about 3 GB to about 1 GB, at a small cost in precision. The adapter itself still trains at full precision.

**QLoRA** is the two together: LoRA on top of a 4-bit base model. That is what makes the fine-tune fit on a free GPU.

**Training and validation loss.** The *loss* measures how far the model's predicted answer is from the correct one (lower is better). *Training loss* comes from the posts the model learns from. *Validation loss* comes from `val.csv`, posts it never trains on. If both fall, the model is learning something general. If training loss keeps falling while validation loss rises, it is memorizing the training posts (**overfitting**).

The loss is computed **on the answer only**, not the prompt: the model should learn to answer, not to write prompts.

## 2. Model and adapter

`load_model` loads a fresh copy of the base model in 4 bits. Each run below starts from a fresh copy, so the tests don't affect the full run.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def load_model():
    """A fresh copy of the base model, with its weights stored in 4 bits."""
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  # NormalFloat4, a 4-bit format designed for model weights
        bnb_4bit_compute_dtype=torch.float16,  # the math still runs in 16 bits (the T4 has no bfloat16)
        bnb_4bit_use_double_quant=True,  # also compress the scaling numbers, saving a little more memory
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=quantization, dtype=torch.float16, device_map="cuda"
    )


LORA_CONFIG = LoraConfig(
    r=16,  # rank: the size of the small matrices
    lora_alpha=32,  # how strongly the adapter counts; twice the rank is a common choice
    lora_dropout=0.05,  # randomly ignore 5% of the adapter's inputs while training, against overfitting
    target_modules="all-linear",  # add an adapter to every attention and feed-forward layer
    task_type="CAUSAL_LM",  # a model that writes text one token at a time
)

## 3. Training data

Each post becomes a prompt and an answer: the exact zero-shot prompt from `src/prompts.py`, and the author's tag as the label. This is one training example as the model sees it. The loss only counts the last line, the label and the end-of-answer marker:

In [ ]:
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv")
val = pd.read_csv(ROOT / "data" / "processed" / "val.csv")
print(f"{len(train):,} training posts, {len(val):,} validation posts\n")


def to_dataset(posts):
    """Turn posts into prompt/answer pairs: the zero-shot prompt, answered with the author's tag."""
    examples = []
    for text, label in zip(posts["text"], posts["label"]):
        examples.append({
            "prompt": build_messages(text),
            "completion": [{"role": "assistant", "content": label}],
        })
    return Dataset.from_list(examples)


example = to_dataset(train.head(1))[0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

Remember the caveat from notebook 01: these labels are the authors' own tags, and "neutral" means the author left the post untagged. The model learns from those tags, noise included.

## 4. Training settings

`make_trainer` sets up one training run. All three runs below (two tests and the full run) use the same settings; only the data, the number of epochs and how often to check validation loss change.

- **Batch size 16:** the model learns from 16 posts at a time. One **step** is one batch, and one **epoch** is one pass through all the training posts.
- **Learning rate 2e-4:** how big each update is. LoRA trains small new matrices, so it uses a higher rate than full fine-tuning. It rises over the first 3% of steps (**warmup**), then falls to zero by the end.
- **16-bit math (`fp16`):** faster and smaller than 32-bit. The T4 doesn't support `bfloat16`, the default in `trl`.
- **Gradient checkpointing:** recomputes some values during training instead of storing them, which saves GPU memory at a small cost in speed.

In [ ]:
def make_trainer(train_posts, val_posts, output_dir, epochs=1, eval_steps=None, save_steps=None, logging_steps=5):
    """One training run: a fresh 4-bit model with a new LoRA adapter, and the settings every run shares."""
    config = SFTConfig(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-4,
        warmup_steps=0.03,  # a number below 1 means a share of all steps
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        max_length=512,  # the longest training example is 368 tokens, so nothing gets cut
        logging_steps=logging_steps,  # how often to record the training loss
        eval_strategy="steps" if eval_steps else "no",
        eval_steps=eval_steps,  # how often to compute the validation loss
        eval_on_start=bool(eval_steps),  # the validation loss before any training, as a starting point
        save_strategy="steps" if save_steps else "no",
        save_steps=save_steps,
        save_total_limit=2,  # keep only the 2 newest checkpoints
        disable_tqdm=False,  # show a progress bar and loss table while training
        report_to="none",  # no online experiment tracker
        seed=SEED,
    )
    trainer = SFTTrainer(
        model=load_model(),
        args=config,
        train_dataset=to_dataset(train_posts),
        eval_dataset=to_dataset(val_posts) if eval_steps else None,
        processing_class=tokenizer,
        peft_config=LORA_CONFIG,
    )
    # trl stores the adapter in bfloat16, which the T4 doesn't support; keep it in 32-bit instead
    for param in trainer.model.parameters():
        if param.requires_grad:
            param.data = param.data.float()
    return trainer


def loss_table(trainer):
    """Training and validation loss at each logged step."""
    log = pd.DataFrame(trainer.state.log_history)
    columns = [column for column in ["loss", "eval_loss"] if column in log]
    table = log.groupby("step")[columns].first()  # a step's training and validation entries become one row
    return table.rename(columns={"loss": "train_loss", "eval_loss": "val_loss"})


def label_posts(model, texts):
    """Ask the fine-tuned model about each post, exactly as notebook 04 did. Returns the raw replies."""
    model.eval()  # switch off training-only behavior such as dropout
    use_greedy_decoding(model)
    return [ask_llm(model, tokenizer, text) for text in texts]

## 5. Convergence test 1: overfit 50 posts

Before spending an hour on the full run, check that training works at all. The model trains on just 50 posts for 20 epochs. It should memorize them: the training loss should fall close to zero, and it should answer all 50 with their training label. If it can't, something is broken (the data, the loss or the settings).

In [ ]:
overfit_posts = train.sample(50, random_state=SEED)
trainer = make_trainer(overfit_posts, None, "/content/overfit", epochs=20)
print(f"Model in GPU memory: {trainer.model.get_memory_footprint() / 1e9:.1f} GB")
trainer.model.print_trainable_parameters()

trainer.train()
loss_table(trainer)

In [ ]:
replies = label_posts(trainer.model, overfit_posts["text"])
memorized = sum(parse_reply(reply) == label for reply, label in zip(replies, overfit_posts["label"]))
print(f"{memorized} of {len(overfit_posts)} posts answered with their training label")

del trainer  # free the GPU for the next run
gc.collect()
torch.cuda.empty_cache()

## 6. Convergence test 2: short run on 1,000 posts

A fresh adapter trains on 1,000 posts for one epoch, with validation loss on 300 validation posts checked along the way. Both losses should fall, and the model's replies should be valid labels. The run is also timed, to estimate how long the full run will take.

In [ ]:
short_train = train.sample(1000, random_state=SEED)
short_val = val.sample(300, random_state=SEED)
trainer = make_trainer(short_train, short_val, "/content/short_run", eval_steps=21)  # 63 steps, so 3 checks

result = trainer.train()
loss_table(trainer)

In [ ]:
replies = label_posts(trainer.model, short_val["text"].head(100))
print(f"Invalid replies: {[parse_reply(reply) for reply in replies].count('INVALID')} of {len(replies)}")
display(pd.Series(replies).value_counts().head())

# Time per step, leaving out the validation checks, then an estimate for the full run
log = pd.DataFrame(trainer.state.log_history)
seconds_per_step = (result.metrics["train_runtime"] - log["eval_runtime"].sum()) / result.global_step
full_steps = math.ceil(len(train) / 16)
full_eval_seconds = 6 * log["eval_runtime"].mean() * len(val) / len(short_val)  # 6 checks on all 3,000 posts
estimate = (full_steps * seconds_per_step + full_eval_seconds) / 60
print(f"{seconds_per_step:.2f} s per step. Full run: {full_steps:,} steps, about {estimate:.0f} minutes.")

del trainer
gc.collect()
torch.cuda.empty_cache()

## 7. Full training run

One epoch over all 30,000 training posts: 1,875 steps of 16 posts. The validation loss on all 3,000 validation posts is checked every 375 steps, so 5 times plus once before training. A checkpoint (the adapter and the training state) is saved to Drive at each check, so a disconnect loses at most 375 steps.

**If Colab disconnects:** reconnect and run the whole notebook again. The two tests repeat (a few minutes), then training resumes from the last checkpoint on Drive.

In [ ]:
CHECKPOINT_DIR = DRIVE_DIR / "checkpoints"
trainer = make_trainer(train, val, CHECKPOINT_DIR, eval_steps=375, save_steps=375, logging_steps=25)

resume = any(CHECKPOINT_DIR.glob("checkpoint-*"))  # True if an earlier run left a checkpoint
result = trainer.train(resume_from_checkpoint=resume)
note = " (since resuming)" if resume else ""
print(f"Training time: {result.metrics['train_runtime'] / 60:.0f} minutes{note}")

Training and validation loss at each check. The full log, every 25 steps, goes to `results/lora_log.csv` for the figures in notebook 06.

In [ ]:
log = loss_table(trainer)
log.to_csv(RESULTS_DIR / "lora_log.csv")
log[log["val_loss"].notna()]

In [ ]:
ADAPTER_DIR = DRIVE_DIR / "lora_adapter"
trainer.save_model(str(ADAPTER_DIR))
print("Adapter files:", sorted(path.name for path in ADAPTER_DIR.iterdir()))

## 8. Predict on the test set

The fine-tuned model labels the 300 test posts with the same prompt, greedy decoding and reply parser as notebook 04 (`ask_llm` and `parse_reply`). It uses the 4-bit base model it was trained with.

In [ ]:
test = load_test()
start = time.time()
replies = label_posts(trainer.model, test["text"])
print(f"{len(test)} posts in {time.time() - start:.0f} s")

save_replies(test["post_id"], replies, "llm_lora")
for name in ["preds_llm_lora.csv", "replies_llm_lora.csv", "lora_log.csv"]:
    shutil.copy(RESULTS_DIR / name, DRIVE_DIR / name)  # Colab's own files vanish when the session ends

pd.Series(replies).value_counts().head(10)  # the most common raw replies

## 9. Results

Macro-F1 for every model, on all posts and on each group. CryptoBERT's score is likely optimistic, since it was trained on the same StockTwits data (see notebook 03).

In [ ]:
MODELS = ["author_tag", "vader", "finbert", "cryptobert", "llm_zeroshot", "llm_fewshot", "llm_lora"]
table = score_table(MODELS)
table.pivot(index="model", columns="group", values="macro_f1").loc[MODELS, ["all", "slang-heavy", "plain"]].round(3)

Invalid-output rate for the fine-tuned model, then F1 for each label on all posts:

In [ ]:
print(f"Invalid replies: {evaluate('llm_lora')['all']['invalid_rate']:.1%}")
pd.DataFrame({model: evaluate(model)["all"]["per_class"]["f1"] for model in MODELS}).T.loc[MODELS, LABELS].round(3)

Confusion matrix on all posts (rows are the hand labels, columns the predictions):

In [ ]:
evaluate("llm_lora")["all"]["confusion"]

## Summary

*Written after the run.*